# 🔍 Execution Audit – Quant Trader
**Workflow**: Load fills → Slippage analysis → SOR venue breakdown → Cost attribution → HTML report

---

In [ ]:
from qtrader.analyst import AnalystSession, RoleContext
import polars as pl
import numpy as np
import matplotlib.pyplot as plt

session = AnalystSession(role=RoleContext.TRADER)

# ---------------------
# 📌 Simulation params – replace with real fills from OMS/broker
# ---------------------
N_FILLS = 500
np.random.seed(42)

## 1. Load Fills (or Simulate)

In [ ]:
# In production: replace with fills from DuckDB or OMS export
# e.g. fills_df = session.load_ohlcv('fills', '1m')  if stored in datalake

venues = np.random.choice(['Coinbase', 'Binance'], size=N_FILLS, p=[0.55, 0.45])
sides  = np.random.choice(['BUY', 'SELL'], size=N_FILLS)
exec_prices = 60000 + np.random.randn(N_FILLS) * 300
intended_prices = exec_prices + np.random.randn(N_FILLS) * (50 if venues == 'Coinbase' else 30).mean()
slippage_bps = (exec_prices - intended_prices) / intended_prices * 10000
qty = np.abs(np.random.randn(N_FILLS) * 0.1 + 0.05)
commission_bps = np.where(venues == 'Coinbase', 6, 4)
total_cost_bps = slippage_bps + commission_bps

fills_df = pl.DataFrame({
    'venue': venues.tolist(),
    'side': sides.tolist(),
    'exec_price': exec_prices.tolist(),
    'intended_price': intended_prices.tolist(),
    'slippage_bps': slippage_bps.tolist(),
    'commission_bps': commission_bps.tolist(),
    'total_cost_bps': total_cost_bps.tolist(),
    'qty': qty.tolist(),
})
print(f"Fills loaded: {len(fills_df)}")
fills_df.head(5)

## 2. Slippage Distribution

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4), facecolor='#0f1117')
for ax in [ax1, ax2]:
    ax.set_facecolor('#0f1117')
    for sp in ax.spines.values(): sp.set_color('#334155')
    ax.tick_params(colors='#94a3b8')
    ax.grid(color='#1e293b', linestyle='--', linewidth=0.5)

slip = fills_df['slippage_bps'].to_numpy()
ax1.hist(slip, bins=50, color='#38bdf8', alpha=0.7, density=True)
ax1.axvline(np.mean(slip), color='#f59e0b', linewidth=2, linestyle='--', label=f'Mean: {np.mean(slip):.2f} bps')
ax1.set_title('Slippage Distribution (bps)', color='#e2e8f0')
ax1.set_xlabel('Slippage (basis points)', color='#94a3b8')
ax1.legend(facecolor='#1e293b', labelcolor='#e2e8f0')

# Slippage by venue
for venue, color in [('Coinbase', '#34d399'), ('Binance', '#f59e0b')]:
    v_slip = fills_df.filter(pl.col('venue') == venue)['slippage_bps'].to_numpy()
    ax2.hist(v_slip, bins=40, alpha=0.6, color=color, density=True, label=venue)
ax2.set_title('Slippage by Venue (bps)', color='#e2e8f0')
ax2.legend(facecolor='#1e293b', labelcolor='#e2e8f0')

plt.tight_layout()
plt.show()
slip_fig = fig

## 3. SOR Venue Cost Attribution

In [ ]:
venue_summary = (
    fills_df.group_by('venue')
    .agg([
        pl.col('slippage_bps').mean().round(3).alias('avg_slippage_bps'),
        pl.col('commission_bps').mean().round(3).alias('avg_commission_bps'),
        pl.col('total_cost_bps').mean().round(3).alias('avg_total_cost_bps'),
        pl.col('qty').sum().round(4).alias('total_qty'),
        pl.count().alias('n_fills'),
    ])
    .with_columns(
        (pl.col('avg_total_cost_bps') * pl.col('total_qty')).alias('total_cost_notional')
    )
    .sort('avg_total_cost_bps')
)
venue_summary

## 4. Cost Component Breakdown

In [ ]:
venues_ = venue_summary['venue'].to_list()
slip_   = venue_summary['avg_slippage_bps'].to_list()
comm_   = venue_summary['avg_commission_bps'].to_list()
x = np.arange(len(venues_))

fig2, ax = plt.subplots(figsize=(8, 4), facecolor='#0f1117')
ax.set_facecolor('#0f1117')
for sp in ax.spines.values(): sp.set_color('#334155')
ax.tick_params(colors='#94a3b8')
ax.grid(color='#1e293b', linestyle='--', linewidth=0.5, axis='y')

ax.bar(x, slip_, label='Slippage', color='#38bdf8', alpha=0.8)
ax.bar(x, comm_, bottom=slip_, label='Commission', color='#f59e0b', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(venues_, color='#e2e8f0')
ax.set_ylabel('Average Cost (bps)', color='#94a3b8')
ax.set_title('Cost Attribution by Venue', color='#e2e8f0')
ax.legend(facecolor='#1e293b', labelcolor='#e2e8f0')
plt.tight_layout()
plt.show()
cost_fig = fig2

## 5. Export HTML Report

In [ ]:
out = session.export_report(
    title='Execution Audit Report',
    sections={
        'Summary': f'Total fills: {len(fills_df)} | Avg slippage: {fills_df["slippage_bps"].mean():.2f} bps',
        'Venue Summary': venue_summary,
        'Slippage Distribution': slip_fig,
        'Cost Attribution': cost_fig,
    },
    path='reports/Execution_Audit.html',
)
print(f'✅ Report saved: {out}')